## 1. Setup and Installation

In [ ]:
# Run this cell in Google Colab to install dependencies
# Skip if running locally with uv
import sys
if 'google.colab' in sys.modules:
    !pip install -q keras torch torchvision python-dotenv datasets transformers huggingface_hub
    print('Dependencies installed!')

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
import numpy as np
import matplotlib.pyplot as plt
import torch

print(f"Keras version: {keras.__version__}")
print(f"Keras backend: {keras.backend.backend()}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Load and Prepare the Dataset

We use CIFAR-10 but filter to only 2 classes: airplane (0) and automobile (1). This keeps training fast while demonstrating the concept.

In [ ]:
# Load CIFAR-10 dataset
(x_train_full, y_train_full), (x_test_full, y_test_full) = keras.datasets.cifar10.load_data()

# Filter for airplane (0) and automobile (1) only
train_mask = (y_train_full.flatten() == 0) | (y_train_full.flatten() == 1)
test_mask = (y_test_full.flatten() == 0) | (y_test_full.flatten() == 1)

x_train = x_train_full[train_mask]
y_train = y_train_full[train_mask].flatten()
x_test = x_test_full[test_mask]
y_test = y_test_full[test_mask].flatten()

# Normalize pixel values to [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

class_names = ["airplane", "automobile"]

print(f"Training set: {x_train.shape}, Labels: {y_train.shape}")
print(f"Test set: {x_test.shape}, Labels: {y_test.shape}")
print(f"Class distribution (train): airplane={np.sum(y_train == 0)}, automobile={np.sum(y_train == 1)}")

In [ ]:
# Visualize some samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i])
    ax.set_title(class_names[y_train[i]])
    ax.axis("off")
plt.suptitle("Sample Training Images", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Build the Feature Extraction Model

We load MobileNetV2 pre-trained on ImageNet, freeze all its layers, and add a custom classification head on top.

In [ ]:
# Load the pre-trained MobileNetV2 base model (without the top classification layer)
base_model = keras.applications.MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(96, 96, 3)  # MobileNetV2 minimum input size is 96x96
)

# Freeze the base model - no weights will be updated during training
base_model.trainable = False

print(f"Base model: {base_model.name}")
print(f"Number of layers: {len(base_model.layers)}")
print(f"Trainable weights: {len(base_model.trainable_weights)}")
print(f"Non-trainable weights: {len(base_model.non_trainable_weights)}")

In [ ]:
# Build the complete model with a custom classification head
inputs = keras.Input(shape=(32, 32, 3))

# Resize CIFAR-10 images (32x32) to 96x96 for MobileNetV2
x = keras.layers.Resizing(96, 96)(inputs)

# Apply MobileNetV2 preprocessing
x = keras.applications.mobilenet_v2.preprocess_input(x)

# Pass through the frozen base model
x = base_model(x, training=False)

# Classification head
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.2)(x)
outputs = keras.layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs, name="feature_extraction_model")
model.summary()

In [ ]:
# Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Count trainable vs non-trainable parameters
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
non_trainable_count = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"\nTrainable parameters: {trainable_count:,}")
print(f"Non-trainable parameters: {non_trainable_count:,}")
print(f"Ratio: {trainable_count / (trainable_count + non_trainable_count) * 100:.2f}% trainable")

## 4. Train the Classification Head

We train only the classification head (the dense layers we added) while keeping the base model frozen.

In [ ]:
# Train the model (only the head will be updated)
history = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=1)
print(f"\nFeature Extraction Model:")
print(f"  Test Loss: {test_loss:.4f}")
print(f"  Test Accuracy: {test_accuracy:.4f}")

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history["loss"], label="Train Loss")
ax1.plot(history.history["val_loss"], label="Val Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Feature Extraction - Loss")
ax1.legend()
ax1.grid(True)

ax2.plot(history.history["accuracy"], label="Train Accuracy")
ax2.plot(history.history["val_accuracy"], label="Val Accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Feature Extraction - Accuracy")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 5. Comparison: Training from Scratch

To appreciate the power of feature extraction, let's train the same architecture from scratch (random weights) and compare.

In [ ]:
# Build the same architecture but with random weights (no pre-training)
scratch_base = keras.applications.MobileNetV2(
    weights=None,  # Random initialization, no pre-trained weights
    include_top=False,
    input_shape=(96, 96, 3)
)

# Build scratch model with the same head
scratch_inputs = keras.Input(shape=(32, 32, 3))
x = keras.layers.Resizing(96, 96)(scratch_inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = scratch_base(x, training=True)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.2)(x)
scratch_outputs = keras.layers.Dense(1, activation="sigmoid")(x)

scratch_model = keras.Model(scratch_inputs, scratch_outputs, name="scratch_model")

scratch_model.compile(
    optimizer=keras.optimizers.Adam(),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print(f"Scratch model trainable parameters: {sum(p.numel() for p in scratch_model.parameters() if p.requires_grad):,}")

In [ ]:
# Train the scratch model for the same number of epochs
scratch_history = scratch_model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# Evaluate scratch model
scratch_loss, scratch_accuracy = scratch_model.evaluate(x_test, y_test, verbose=1)
print(f"\nScratch Model:")
print(f"  Test Loss: {scratch_loss:.4f}")
print(f"  Test Accuracy: {scratch_accuracy:.4f}")

print(f"\n--- Comparison ---")
print(f"Feature Extraction: {test_accuracy:.4f}")
print(f"Training from Scratch: {scratch_accuracy:.4f}")
print(f"Improvement: {(test_accuracy - scratch_accuracy) * 100:.2f} percentage points")

In [ ]:
# Compare learning curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history["val_accuracy"], label="Feature Extraction", marker="o")
ax1.plot(scratch_history.history["val_accuracy"], label="From Scratch", marker="s")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Validation Accuracy")
ax1.set_title("Validation Accuracy Comparison")
ax1.legend()
ax1.grid(True)

ax2.plot(history.history["val_loss"], label="Feature Extraction", marker="o")
ax2.plot(scratch_history.history["val_loss"], label="From Scratch", marker="s")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Validation Loss")
ax2.set_title("Validation Loss Comparison")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 6. Visualize Predictions

In [ ]:
# Make predictions on test set
predictions = model.predict(x_test[:20])
predicted_classes = (predictions.flatten() > 0.5).astype(int)

# Display predictions
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_test[i])
    pred_label = class_names[predicted_classes[i]]
    true_label = class_names[y_test[i]]
    confidence = predictions[i][0] if predicted_classes[i] == 1 else 1 - predictions[i][0]
    color = "green" if pred_label == true_label else "red"
    ax.set_title(f"Pred: {pred_label}\nTrue: {true_label}\nConf: {confidence:.2f}", color=color, fontsize=9)
    ax.axis("off")
plt.suptitle("Feature Extraction Model Predictions", fontsize=14)
plt.tight_layout()
plt.show()